# ROSE-1 SVC — Classical Baseline Grid Search

Two-stage grid search for the adaptive-threshold classical baseline: first
CLAHE `(clip_limit, tile_size)` with segmentation held at a default, then
`(block_size, C)` with preprocessing locked to the winner of stage 1. Each
config is scored with 5-fold CV on the training set only (`evaluate_config_cv`);
the held-out 9 test images are touched exactly once, at the end, with the
locked-in best params. Winning params (`clip=2.0, tile=4, block=101, C=-30`)
are what `src/classical_baseline.py` and the other ROSE notebooks use.


In [1]:
import numpy as np
import cv2
from pathlib import Path
import sys
sys.path.append('/users/egottfri/code/octa-segmentation/src')
from evaluate import dice

ROSE1 = Path("/files22_lrsresearch/ENG_Lee-Lab_Shared/group/data/public/rose_dataset/ROSE-1")

# TRAINING images only — never touch test during search
train_imgs = sorted((ROSE1 / "SVC/train/img").glob("*.tif"))
train_masks = sorted((ROSE1 / "SVC/train/gt").glob("*.tif"))

def evaluate_config(clip_limit, tile_size, block_size, C, img_paths, mask_paths):
    """Apply preprocessing + adaptive threshold, return mean Dice over the image set."""
    scores = []
    for img_path, mask_path in zip(img_paths, mask_paths):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        gt = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        # Preprocessing
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
        proc = clahe.apply(img)
        proc = cv2.medianBlur(proc, 3)

        # Segmentation (adaptive threshold)
        pred = cv2.adaptiveThreshold(proc, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, block_size, C)

        scores.append(dice(pred, gt))
    return np.mean(scores)

In [5]:
from sklearn.model_selection import KFold

def evaluate_config_cv(clip, tile, block, C, img_paths, mask_paths, n_splits=5):
    """
    Evaluate a config using k-fold CV on the training data.
    Returns mean and std of validation Dice across folds.
    """
    img_paths = list(img_paths)
    mask_paths = list(mask_paths)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_scores = []
    for _, val_idx in kf.split(img_paths):
        # Only evaluate on this fold's validation images
        val_scores = []
        for i in val_idx:
            img = cv2.imread(str(img_paths[i]), cv2.IMREAD_GRAYSCALE)
            gt = cv2.imread(str(mask_paths[i]), cv2.IMREAD_GRAYSCALE)

            clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
            proc = clahe.apply(img)
            proc = cv2.medianBlur(proc, 3)
            pred = cv2.adaptiveThreshold(proc, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                         cv2.THRESH_BINARY, block, C)
            val_scores.append(dice(pred, gt))

        fold_scores.append(np.mean(val_scores))

    return np.mean(fold_scores), np.std(fold_scores)

In [7]:
# ============================================================
# SEARCH 1: Preprocessing parameters (CLAHE clip_limit, tile_size)
# Hold segmentation fixed at a reasonable default while searching preprocessing
# ============================================================
print("="*60)
print("SEARCH 1: Preprocessing parameters")
print("="*60)

clip_limits = [1.0, 2.0, 3.0, 4.0]
tile_sizes = [4, 8, 16]

# Fixed segmentation defaults during preprocessing search
DEFAULT_BLOCK = 71
DEFAULT_C = -30

prep_results = []
for clip, tile in itertools.product(clip_limits, tile_sizes):
    mean_dice, std_dice = evaluate_config_cv(clip, tile, DEFAULT_BLOCK, DEFAULT_C,
                                             train_imgs, train_masks)
    prep_results.append({'clip_limit': clip, 'tile_size': tile,
                         'dice': mean_dice, 'std': std_dice})
    print(f"clip={clip}, tile={tile}: Dice={mean_dice:.4f} ± {std_dice:.4f}")

# Find best preprocessing
prep_df = pd.DataFrame(prep_results).sort_values('dice', ascending=False)
best_prep = prep_df.iloc[0]
print(f"\nBest preprocessing: clip={best_prep['clip_limit']}, "
      f"tile={best_prep['tile_size']} (Dice={best_prep['dice']:.4f})")

best_clip = best_prep['clip_limit']
best_tile = int(best_prep['tile_size'])

SEARCH 1: Preprocessing parameters
clip=1.0, tile=4: Dice=0.6763 ± 0.0103
clip=1.0, tile=8: Dice=0.6751 ± 0.0102
clip=1.0, tile=16: Dice=0.6726 ± 0.0097
clip=2.0, tile=4: Dice=0.6783 ± 0.0101
clip=2.0, tile=8: Dice=0.6776 ± 0.0096
clip=2.0, tile=16: Dice=0.6767 ± 0.0095
clip=3.0, tile=4: Dice=0.6740 ± 0.0098
clip=3.0, tile=8: Dice=0.6743 ± 0.0096
clip=3.0, tile=16: Dice=0.6702 ± 0.0095
clip=4.0, tile=4: Dice=0.6668 ± 0.0093
clip=4.0, tile=8: Dice=0.6675 ± 0.0092
clip=4.0, tile=16: Dice=0.6636 ± 0.0091

Best preprocessing: clip=2.0, tile=4.0 (Dice=0.6783)


In [9]:
# ============================================================
# SEARCH 2: Segmentation parameters (block_size, C)
# Now hold preprocessing fixed at the best found above
# ============================================================
print("="*60)
print("SEARCH 2: Segmentation parameters (using best preprocessing)")
print("="*60)

block_sizes = [11, 21, 31, 51, 71, 101]
c_values = [-50, -30, -20, -10, -5, 0, 2, 5]

seg_results = []
for block, C in itertools.product(block_sizes, c_values):
    mean_dice, std_dice = evaluate_config_cv(best_clip, best_tile, block, C,
                                             train_imgs, train_masks)
    seg_results.append({'block_size': block, 'C': C,
                        'dice': mean_dice, 'std': std_dice})
    print(f"block={block}, C={C}: Dice={mean_dice:.4f} ± {std_dice:.4f}")

seg_df = pd.DataFrame(seg_results).sort_values('dice', ascending=False)
best_seg = seg_df.iloc[0]
print(f"\nBest segmentation: block={best_seg['block_size']}, "
      f"C={best_seg['C']} (Dice={best_seg['dice']:.4f})")

best_block = int(best_seg['block_size'])
best_C = int(best_seg['C'])

SEARCH 2: Segmentation parameters (using best preprocessing)
block=11, C=-50: Dice=0.3758 ± 0.0133
block=11, C=-30: Dice=0.5809 ± 0.0067
block=11, C=-20: Dice=0.6082 ± 0.0076
block=11, C=-10: Dice=0.5842 ± 0.0093
block=11, C=-5: Dice=0.5563 ± 0.0116
block=11, C=0: Dice=0.5171 ± 0.0148
block=11, C=2: Dice=0.4966 ± 0.0157
block=11, C=5: Dice=0.4702 ± 0.0171
block=21, C=-50: Dice=0.5756 ± 0.0090
block=21, C=-30: Dice=0.6495 ± 0.0081
block=21, C=-20: Dice=0.6429 ± 0.0074
block=21, C=-10: Dice=0.6070 ± 0.0084
block=21, C=-5: Dice=0.5803 ± 0.0108
block=21, C=0: Dice=0.5471 ± 0.0134
block=21, C=2: Dice=0.5308 ± 0.0142
block=21, C=5: Dice=0.5070 ± 0.0155
block=31, C=-50: Dice=0.6193 ± 0.0105
block=31, C=-30: Dice=0.6664 ± 0.0090
block=31, C=-20: Dice=0.6541 ± 0.0077
block=31, C=-10: Dice=0.6181 ± 0.0082
block=31, C=-5: Dice=0.5929 ± 0.0102
block=31, C=0: Dice=0.5618 ± 0.0123
block=31, C=2: Dice=0.5470 ± 0.0132
block=31, C=5: Dice=0.5240 ± 0.0145
block=51, C=-50: Dice=0.6427 ± 0.0115
block=51, 

In [10]:
# Locked-in best parameters from the grid search
BEST_CLIP = 2.0
BEST_TILE = 4
BEST_BLOCK = 101
BEST_C = -30

# TEST images (the held-out 9 — never used in search)
test_imgs = sorted((ROSE1 / "SVC/test/img").glob("*.tif"))
test_masks = sorted((ROSE1 / "SVC/test/gt").glob("*.tif"))

def segment_image(img, clip, tile, block, C, use_morph=True):
    """Full pipeline: preprocess + adaptive threshold + optional morphology."""
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
    proc = clahe.apply(img)
    proc = cv2.medianBlur(proc, 3)
    pred = cv2.adaptiveThreshold(proc, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, block, C)
    if use_morph:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        pred = cv2.morphologyEx(pred, cv2.MORPH_CLOSE, kernel)
    return pred

# Evaluate on each test image, collect full metrics
test_metrics = []
for img_path, mask_path in zip(test_imgs, test_masks):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    pred = segment_image(img, BEST_CLIP, BEST_TILE, BEST_BLOCK, BEST_C)
    m = evaluate(pred, gt)   # returns dict: dice, iou, sensitivity, etc.
    test_metrics.append(m)

# Average across the 9 test images
final = {k: np.mean([m[k] for m in test_metrics]) for k in test_metrics[0]}
std = {k: np.std([m[k] for m in test_metrics]) for k in test_metrics[0]}

print("="*60)
print("CLASSICAL BASELINE — TEST SET RESULTS (best params, no leakage)")
print(f"Params: clip={BEST_CLIP}, tile={BEST_TILE}, block={BEST_BLOCK}, C={BEST_C}")
print("="*60)
for k in final:
    print(f"{k}: {final[k]:.4f} ± {std[k]:.4f}")

CLASSICAL BASELINE — TEST SET RESULTS (best params, no leakage)
Params: clip=2.0, tile=4, block=101, C=-30
dice: 0.6714 ± 0.0278
iou: 0.5060 ± 0.0316
sensitivity: 0.6980 ± 0.0665
specificity: 0.9167 ± 0.0110
precision: 0.6518 ± 0.0300
